In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("w13Blab.ipynb")

---

<h3><center>E7 -  Introduction to Programming for Scientists and Engineers</center></h3>

<h2><center>Lab session #13-B <br></center></h2>

<h1><center>Temperature distribution on a square plate<br></center></h1>

---

In [ ]:
from resources.hashutils import *
import numpy as np
import matplotlib.pyplot as plt

A scientific experiment requires that the temperature distribution on a thin square metallic plate be carefully monitored. However the physical constraints of the experiment only allow that temperature sensors be placed at the four corners of the plate (red nodes in the figure below). To deal with this problem, the researchers have decided to *estimate* the temperatures at the remaining 12 points on a 4x4 grid (white nodes in the figure), by assuming that the temperature of any unmeasured node is the average of its immediate neighbors. So for example, 

+ the temperature at node 1 equals the average of the temperatures of nodes 0, 2, and 5.
+ the temperature at node 5 equals the average of the temperatures of nodes 4, 1, 6, and 9.

Note that this is only true for white nodes; the temperatures at red nodes are prescribed and may not follow this rule. 

In this exercise we will use the tools of linear algebra to compute the twelve missing temperatures. 

<img src="resources/grid.png" width="400" />


## Part 1: Node coordinate to index map

Each node is identified by either its coordinates (e.g. (2,1)) or its index (e.g 9). These are shown in the figure. The indexes are the integers inside each of the circles, and the coordinates are the tuples shown next to the nodes. Both of these identifiers will be useful, and hence we will need to translate between the two representations. 

Write two functions, called `coord2index(coord)` and `index2coord(index)`, which respectively translate from a node's coordinate to its index and vice-versa. 

+ `coord2index(coord)` takes a coordinate tuple and returns an integer index.
+ `index2coord(index)` takes an integer index and returns a coordinate tuple. 

**Hint**: While it is possible to write these functions using long if-elif clauses, it is also possible to do it with a single line of code. Try it!

In [ ]:
...

In [ ]:
# test your code

print( coord2index((2,1)) )
print( index2coord(9) )

In [ ]:
grader.check("p1")

## Part 2: Measured and unmeasured nodes

Create two Python tuples:
+ `nodes_measured`, containing the indexes of measured nodes, and
+ `nodes_unmeasured`, containing the indexes of unmeasured nodes.

Each of these tuples should be in ascending order. 

**Note**: 
+ We are using Python tuples instead of NumPy arrays because we will not perform significant mathematical operations on these values. 
+ We are using tuples instead of lists because the values should remain fixed.

In [ ]:
...

In [ ]:
grader.check("p2")

## Part 3: Dictionary of measured temperatures

Create a dictionary called `index2temp` whose keys are the indexes of measured nodes, and whose values are their respective temperatures (shown in the figure).

In [ ]:
...

In [ ]:
grader.check("p3")

## Part 4: Get a node's neighbors

Create a function called `get_neighbors(index)` that, for a given node index, returns a *set* containing the indexes of its neighbors.
This set will have 4 elements if the node is an interior node, 3 if it is a boundary node, and 2 if it is a corner node.

**Hint**: Consider all four directions around the indexed node. 

In [ ]:
...

In [ ]:
grader.check("p4")

################################

## Part 5: Matrix form for the system of equations

We will use $T_q$ to denote the temperature at the node with index $q$. Then $T_0$, $T_3$, $T_{12}$, and $T_{15}$ are given quantities, and the unknowns in our problem are the temperatures of the 12 unmeasured nodes: $T_1, T_2, T_4, ..., T_{14}$. 

There are 12 equations in our system of equations, corresponding to the 12 unknowns. Each equation determines a node's temperature as the average of its neighbors. So for example, for node 5,

\begin{equation*}
T_5 = \frac{1}{4}\Bigl( T_1 + T_4 + T_6 + T_9 \Bigr)
\end{equation*}

Or equivalently, 

\begin{equation*}
T_5 - \frac{1}{4}\Bigl( T_1 + T_4 + T_6 + T_9 \Bigr) = 0
\end{equation*}

For boundary nodes, one of the temperatures in the formula is given, and therefore should remain on the right hand side of the equation. For example, for node 4:

\begin{equation*}
T_4 - \frac{1}{3}\Bigl( T_5 + T_8 \Bigr) = \frac{1}{3} T_0
\end{equation*}


In order to solve the system of equations, we must first put it into the matrix form $Ax=b$, where $A$ is a $12\times 12$ matrix and $b$ is a $12\times 1$ column vector, and $x$ is the vector of unknown temperatures arranged in order of increasing index.

\begin{equation*}
x \;=\; \begin{bmatrix}x_0\\x_1\\x_2\\x_3\\ \vdots\\x_{11}\end{bmatrix} \;=\; \begin{bmatrix}T_1\\T_2\\T_4\\T_5\\ \vdots\\T_{14}\end{bmatrix}
\end{equation*}

The equation for nodes 4 and 5 appear respectively in the third and fourth rows of $A$:

<img src=resources/eqns.png width=700>


**Task**: Write a function called `create_matrix_form` that takes no inputs and returns the `A` and `b`, where `A` is a 2D NumPy array corresponding to the matrix $A$, and `b` is a 1D NumPy array corresponding to the column vector $b$.

**Notes**
+ `create_matrix_form()` should take no inputs, even though it will use `nodes_unmeasured` and `nodes_measured`. These are accessible because they are in the global scope. 
+ Both `A` and `b` will contain a significant number of zeros. Hence we sugget initializing them with `np.zeros`, and only filling in the non-zero values. 
+ We suggest to first try to do this "from scratch". However we have included a template notebook in the `resources` folder if you'd like some guidance.

**Hints**
+ The `seq.index(v)` method of a Python sequence returns the index of the first occurrence of `v` in `seq`.
+ Remember that the keyword `in` can be used to check whether an item exists in a Python collection. For example the following code will print "hello".
```python
Z = {1,2,3}
a = 1
if a in Z:
    print("hello")
```

In [ ]:
def create_matrix_form():

    A = ...
    b = ...

    ...

    return A, b

In [ ]:
grader.check("p5")

## Part 6: Check the ranks of $A$ and $(A|b)$

In lecture we covered conditions involving the rank of $A$ and the rank of the extended matrix $(A|b)$ for determining whether a system of linear equations has 0, 1, or infinitely many solutions.  Compute the ranks of these matrices using [np.linalg.matrix_rank](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html) and [np.hstack](https://numpy.org/doc/stable/reference/generated/numpy.hstack.html), and assign them to `rankA` and `rankAb`. Then observe the values that you got and assign to `num_solutions` the number of solutions that problem should have (0, 1, or [np.inf](https://numpy.org/devdocs/reference/constants.html#numpy.inf))

In [ ]:
A, b = create_matrix_form()
rankA = ...
rankAb = ...
num_solutions = ...

In [ ]:
grader.check("p6")

## Part 7: Solve for $x$

Given what was found in part 6, we can assert that the unique solution to the problem is given by:
\begin{equation*}
x = A^{-1}b
\end{equation*}

Write code to find this value of `x`.

**Hints**:
+ [`np.linalg.inv`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.inv.html)


In [ ]:
x = ...

In [ ]:
grader.check("p7")

## Part 8: Compute T

Organize the measured and unmeasured (but now estimated) temperature values into a 4x4 matrix called `T`. The row and column indices for the entries of `T` are simply the coordinates of the nodes from the initial figure.

In [ ]:
T = ...
...
...
...

In [ ]:
grader.check("p8")

## Part 9. Plot

The next cell generates a surface plot of the temperatures across the plate. This part is not graded, it is simply so that you can see the result. You can add `%matplotlib widget` if you want to rotate the plot, but remember to remove that line before submitting, or else it will confuse the autograder. 

In [ ]:
rows, cols = np.meshgrid(range(4),range(4))
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.scatter(rows, cols,T,marker='o',c='m',s=60)
ax.plot_surface(rows, cols,T,cmap='coolwarm',alpha=0.8)

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

Make sure you submit the .zip file to Gradescope.

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)